# Guía de Trabajo Práctico 1 

**Materia**: Aprendizaje Profundo Basado en la Física (optativa del Instituto Balseiro)

**Docente**: José I. Robledo

**Edición**: abril-mayo 2026

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jorobledo/apbf/blob/main/APBF/practicos/Guia_Semana_01_Fundamentos_Python_y_ML.ipynb)

## Parte 1 - Introducción a la programación orientada a objetos en Python             


## Imports

In [ ]:
from __future__ import annotations

from random import Random
import matplotlib.pyplot as plt
import numpy as np

---

## Ejercicio 1 - Clase `Medicion`

Supongamos que queremos representar una medicion simple de un experimento. Para esto utilizaremos una clase y veremos la ventaja de la programación orientada a objetos que ofrece Python. Supongamos que medimos el valor de una variable en su correspondiente unidad (podría ser peso en Kg, temperatura en Kelvin, largo en cm, lo que se te ocurra).

1. Crear una *clase* `Medicion` con *atributos*:
   - `nombre_variable` (str)
   - `valor` (float)
   - `unidad` (str)
2. Implementar:
   - `__repr__` para que la instancia sea facil de inspeccionar
   - `es_positiva()` que devuelva `True` si `valor` es un numero no negativo
   - `to_dict()` que devuelva un diccionario con la informacion de la medición
3. Crear al menos 3 mediciones distintas y mostrarlas.


In [ ]:
# TODO: definir la clase Medicion
class Medicion:
    def __init__(self, nombre: str, valor: float, unidad: str):
        self.nombre = nombre
        self.valor = valor
        self.unidad = unidad

    def __repr__(self):
        return f"Medicion(nombre='{self.nombre}', valor={self.valor}, unidad='{self.unidad}')"
    
    def es_positiva(self) -> bool:
        return self.valor > 0
    
    def to_dict(self) -> dict:
        return {"nombre": self.nombre, "valor": self.valor, "unidad": self.unidad}

# TODO: crear 3 instancias
medicion1 = Medicion("Temperatura", 25.0, "°C")
medicion2 = Medicion("Presión", 1013.25, "hPa")
medicion3 = Medicion("Humedad", 60.0, "%")

# TODO: imprimirlas y verificar su correcto funcionamiento
print(medicion1.__repr__())
print(medicion1.to_dict())

---

## Ejercicio 2 - Clase `SerieDeMediciones`

Ahora queremos agrupar mediciones de una misma variable y exponer métodos útiles para la serie de mediciones. Además, queremos asegurar que sólo podamos agrupar mediciones del mismo tipo.

1. Crear una clase `SerieDeMediciones` que reciba:
   - `nombre`
   - una lista opcional de objetos `Medicion`
2. Implementar los *métodos*:
   - `agregar(medicion)`: agrega instancia de `Medición` a la lista de mediciones.
   - `eliminar(medicion)`: elimina una medición de la lista a partir de su índice posicional.
   - `valores()`: devuelve una lista con los valores numericos de todas las mediciones en la lista. 
   - `promedio()`: calcula el promedio muestral de los valores numéricos de la lista de mediciones como $\sum_{i=1}^{N} x_i /N$.
   - `desvio_estandar()`: calcula el desvío estándar muestral como $\sum_{i=1}^{N-1} (x_i - \bar x)^2/(N-1)$
   - `minimo()`: Calcula el mínimo de los valores numéricos de la lista de mediciones.
   - `maximo()`: Calcula el máximo de los valores numéricos de la lsita de mediciones.
   - `resumen()`: devuelve un diccionario con la cantidad de mediciones, el valor promedio, el mínimo y el máximo
3. Verificar que no se puedan agregar objetos de tipo incorrecto.
4. Crear una serie con al menos 5 mediciones de temperatura y probar todos los *métodos* implementados.


In [ ]:
# TODO: definir SerieDeMediciones
class SerieDeMediciones:
    def __init__(self, nombre: str, mediciones: list[Medicion] = None):
        self.nombre = nombre
        self.mediciones = mediciones if mediciones is not None else []
    
    def __repr__(self):
        return f"SerieDeMediciones(nombre='{self.nombre}', mediciones={self.mediciones})"

    def agregar_medicion(self, medicion: Medicion):
        self.mediciones.append(medicion)
    
    def eliminar_medicion(self, indice: int):
        if 0 <= indice < len(self.mediciones):
            self.mediciones.pop(indice)
    
    def valores(self) -> list[float]:
        return [medicion.valor for medicion in self.mediciones]
    
    def promedio(self) -> float:
        valores = self.valores()
        return sum(valores) / len(valores) if valores else 0.0
    
    def desviacion_estandar(self) -> float:
        valores = self.valores()
        if not valores:
            return 0.0
        media = self.promedio()
        varianza = sum((valor - media) ** 2 for valor in valores) / len(valores)
        return varianza ** 0.5
    
    def minimo(self) -> float:
        valores = self.valores()
        return min(valores) if valores else 0.0
    
    def maximo(self) -> float:
        valores = self.valores()
        return max(valores) if valores else 0.0
    
    def resumen(self) -> dict:
        return {
            "promedio": self.promedio(),
            "desviacion_estandar": self.desviacion_estandar(),
            "minimo": self.minimo(),
            "maximo": self.maximo()
        }
    
    
# TODO: crear una serie de temperaturas

def temp_series():
    random = Random(42)
    mediciones = [Medicion("Temperatura", random.uniform(15.0, 30.0), "°C") for _ in range(50000)]
    return SerieDeMediciones("Serie de Temperaturas", mediciones)

serie1 = temp_series()
print(serie1.valores())


# TODO: probar los métodos implementados
# serie1.agregar_medicion(Medicion("Temperatura", 22.5, "°C"))
# print(serie1.valores())
# serie1.eliminar_medicion(0)
print(serie1.valores())

print(serie1.promedio())
print(serie1.desviacion_estandar())
serie1.resumen()


---

## Ejercicio 3 - Clase `DatasetRegresion`

La dependencia de la presión de vapor de saturación del agua con la temperatura puede aproximarse, bajo la hipótesis de vapor ideal y calor latente constante, mediante la forma integrada de la ecuación de Clausius–Clapeyron:

$$
\ln P = -\frac{L}{R_v T} + C
$$

donde:
- \(P\) es la presión de vapor de saturación (Pa),
- \(T\) es la temperatura absoluta (K),
- \(L\) es la entalpía de vaporización (J/kg),
- \(R_v\) es la constante de gas del vapor de agua (J/(kg·K)),
- \(C\) es una constante relacionada con el estado.

Reordenando, se puede obtener una expresión aproximada para \(T\) en función de \(P\):

$$
T \approx \frac{L}{R_v (C - \ln P)}
$$

Usaremos esta relación para generar datos sintéticos de temperatura dependientes de la presión, y añadiremos un término adicional de ruido para hacer el dataset más realista y adecuado para un problema de regresión.  Observando la tendencia de los datos, decidimos hacer una regresión 1D. Para el problema de regresión, necesitamos que cada muestra tenga:
- una valor de `presion` como variable medida
- un valor de temperatura `T` como respuesta

1. Crear una clase `DatasetRegresion` que almacene los datos. Cada muestra puede representarse como diccionario o como objeto propio.
2. Implementar:
   - `agregar_muestra(x, y)`
   - `features()` que devuelva la lista de `x`
   - `targets()` que devuelva la lista de `y`
   - `__len__()`
   - `train_test_split(test_size=0.2, seed=42)` que separe al conjunto de datos en una parte de entrenamiento y otra de prueba y devuelva dos instancias de `DatasetRegression` de las particiones.
3. Revisar la generación de dataset sintético con una regla aproximada tipo `T = L/(R_v*(C - np.log(P_pa + 1e-8))) - 273.15 + ruido` para presiones entre 850hPa y 1050hPa usando `ruido` aleatorio uniforme entre -1.5 y 1.5. Para toda instancia aleatoria, tener en cuenta una semilla con `Random(seed)`.
4. Separar el conjunto de datos generados en 75% para entrenamiento y 25% para prueba y mostrar cuantos elementos hay en cada uno.

In [ ]:
# TODO: definir DatasetRegresion
class DataRegression:
    def __init__(self, presiones: list[float], temperaturas: list[float]):
        self.presiones = presiones
        self.temperaturas = temperaturas
    def agregar_muestra(self, presion: float, temperatura: float):
        self.presiones.append(presion)
        self.temperaturas.append(temperatura)
    
    def features(self) -> list[float]:
        return self.presiones
    
    def targets(self) -> list[float]:
        return self.temperaturas
    
    def __len__(self):
        return len(self.presiones)
    
    def train_test_split(self, test_size: float = 0.2, seed = 42) -> tuple[DataRegression, DataRegression]:
        rng = Random(seed)
        indices = list(range(len(self)))
        rng.shuffle(indices)
        split_idx = int(len(self) * (1 - test_size))
        train_indices = indices[:split_idx]
        test_indices = indices[split_idx:]

        train_data = DataRegression(
            presiones=[self.presiones[i] for i in train_indices],
            temperaturas=[self.temperaturas[i] for i in train_indices]
        )
        test_data = DataRegression(
            presiones=[self.presiones[i] for i in test_indices],
            temperaturas=[self.temperaturas[i] for i in test_indices]
        )
        return train_data, test_data
    

# Generar datos sinteticos reproducibles
rng = Random(7)
presiones, temperaturas = [], []
for _ in range(100):
    presion = rng.uniform(850, 1050)
    presiones.append(presion)
    ruido = rng.uniform(-1.5, 1.5)

    # Relación inspirada en Clausius–Clapeyron (vapor de agua)
    P_pa = presion * 100.0
    L = 2.26e6  # J/kg, latente de vaporización del agua
    R_v = 461.5  # J/(kg·K), gas constante del vapor de agua
    C = 20.0

    # Temperatura (K) del vapor saturado igualada con la ecuación de Clausius-Clapeyron
    # y convertida a °C.
    T_k = L / (R_v * (C - np.log(P_pa + 1e-8)))
    temperatura = T_k - 273.15 +  ruido
    temperaturas.append(temperatura)

# TODO: hacer split train/test
dataset = DataRegression(presiones, temperaturas)
train_data, test_data = dataset.train_test_split(test_size=0.2, seed=7)
# TODO: imprimir tamaños y algun ejemplo
print(f"Train size: {len(train_data)}, Test size: {len(test_data)}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(presiones, temperaturas, alpha=0.7)
plt.xlabel('Presión (hPa)')
plt.ylabel('Temperatura (°C)')
plt.title('Temperatura vs Presión')
plt.grid(True)
plt.show()

---

## Ejercicio 4 - Modelo con interfaz `fit` / `predict`

Queremos una clase que aprenda una recta a partir de datos 1D usando una fórmula cerrada de mínimos cuadrados.

1. Crear una clase `RegresionLinealSimple` con *atributos* `pendiente` e `intercepto`.
2. Implementar:
   - `fit(xs, ys)` para estimar la recta `y = m*x + b` utilizando las ecuaciones de mínimos cuadrados.
   - `predict(xs)` que reciba una lista de entradas y devuelva una lista de predicciones
   - `predict_one(x)` que reciba un valor escalar y devuelva una predicción puntual
3. Entrenar el modelo sobre el dataset train del ejercicio anterior.
4. Mostrar `pendiente` e `intercepto` aprendidos.
5. Graficar la predicción sobre el conjunto de datos de entrenamiento, y luego la predicción usando los datos de prueba.
6. Opcional: Realizar el ejercicio utilizando [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) de `scikit-learn`.
   
### Ecuaciones de Mínimos Cuadrados

- Media:
  $$\bar x = \frac{1}{n} \sum_{i=1}^{n} x_i, \qquad \bar y = \frac{1}{n} \sum_{i=1}^{n} y_i$$

- Pendiente:
  $$m = \frac{\sum_{i=1}^{n} (x_i - \bar x)(y_i - \bar y)}{\sum_{i=1}^{n} (x_i - \bar x)^2}$$

- Intercepto:
  $$b = \bar y - m\,\bar x$$

- Predicción:
  $$\hat y = m x + b$$


In [ ]:
# TODO: definir RegresionLinealSimple
class RegresionLinealSimple:
    def __init__(self):
        self.coef_ = 0.0
        self.intercept_ = 0.0
    
    def fit(self, X: list[float], Y: list[float]):
        x_mean = np.mean(X)
        y_mean = np.mean(Y)
        numerator = sum((x - x_mean) * (y - y_mean) for x, y in zip(X, Y))  
        denominator = sum((x - x_mean) ** 2 for x in X)
        self.coef_ = numerator / denominator if denominator != 0 else 0.0
        self.intercept_ = y_mean - self.coef_ * x_mean

    def predict(self, X: list[float]) -> list[float]:
        return [self.intercept_ + self.coef_ * x for x in X]
    
    def predict_one(self, x: float) -> float:
        return self.intercept_ + self.coef_ * x
    

# TODO: entrenar con train.features() y train.targets()
model = RegresionLinealSimple()
model.fit(train_data.features(), train_data.targets())
# TODO: imprimir parámetros y algunas predicciones
print(f"Coeficiente: {model.coef_}, Intercepto: {model.intercept_}")
print(f"Predicción para presión 900 hPa: {model.predict_one(900)} °C")
# TODO: graficar predicción con datos de entrenamiento y de prueba.
plt.figure(figsize=(7, 4))
plt.scatter(train_data.features(), train_data.targets(), alpha=0.7, label='Train')
plt.scatter(test_data.features(), test_data.targets(), alpha=0.7, label='Test')
x_line = np.linspace(min(dataset.features()), max(dataset.features()), 100)
y_line = model.predict(x_line)
plt.plot(x_line, y_line, color='red', label='Regresión Lineal')
plt.xlabel('Presión (hPa)')
plt.ylabel('Temperatura (°C)')
plt.title('Regresión Lineal Simple')
plt.legend()
plt.grid(True)
plt.show()

## Ejercicio 5 - Clase `MetricasRegresion`

Vamos a separar la lógica del modelo de la lógica de evaluación.

1. Crear una clase `MetricasRegresion` con métodos estáticos:

   * `mse(y_true, y_pred)`
   * `mae(y_true, y_pred)`
   * `r2(y_true, y_pred)`

2. Evaluar el modelo del ejercicio 4 sobre el conjunto de test.

3. Mostrar los valores obtenidos con 4 decimales.

### Definición de métricas

Las métricas de regresión se definen de la siguiente manera:

* **Error Cuadrático Medio (MSE)**
$$
MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

* **Error Absoluto Medio (MAE)**
$$
MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|
$$

* **Coeficiente de Determinación (R²)**
$$
R^2 = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}
$$

donde:

* $y_i$ son los valores reales
* $\hat{y}_i$ son las predicciones del modelo
* $\bar{y}$ es el promedio de los valores reales
* $n$ es la cantidad de muestras


In [ ]:
# TODO: definir MetricasRegresion
class MetricasRegresion:
    @staticmethod
    def mse(y_true: list[float], y_pred: list[float]) -> float:
        return np.mean([(yt - yp) ** 2 for yt, yp in zip(y_true, y_pred)])
    
    @staticmethod
    def mae(y_true: list[float], y_pred: list[float]) -> float:
        return np.mean([abs(yt - yp) for yt, yp in zip(y_true, y_pred)])
    
    @staticmethod
    def r2_score(y_true: list[float], y_pred: list[float]) -> float:
        y_mean = np.mean(y_true)
        ss_total = sum((yt - y_mean) ** 2 for yt in y_true)
        ss_residual = sum((yt - yp) ** 2 for yt, yp in zip(y_true, y_pred))
        return 1 - ss_residual / ss_total if ss_total != 0 else 0.0
# TODO: evaluar en test
y_true = test_data.targets()
y_pred = model.predict(test_data.features())
mse = MetricasRegresion.mse(y_true, y_pred)
mae = MetricasRegresion.mae(y_true, y_pred)
r2 = MetricasRegresion.r2_score(y_true, y_pred)
# TODO: imprimir métricas con formato
print(f"MSE: {mse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")

---

## Ejercicio 6 - Clase `Experimento`

Crear una clase de más alto nivel que coordine dataset, modelo y resultados.

1. Crear una clase `ExperimentoRegresion` que reciba:
   - `nombre`
   - `dataset_train`
   - `dataset_test`
   - `modelo`
2. Implementar un método `ejecutar()` que:
   - entrene el modelo
   - compute predicciones en train y test
   - guarde un atributo `historial` como lista de diccionarios con métricas
3. Implementar `reporte()` que devuelva un string legible con el resumen del experimento.
4. Ejecutar el experimento y mostrar el reporte.

In [ ]:
# TODO: definir ExperimentoRegresion
class ExperimentoRegresion:
    def __init__(self, nombre: str, dataset_train: DataRegression, dataset_test: DataRegression, model):
        self.nombre = nombre
        self.dataset_train = dataset_train
        self.dataset_test = dataset_test
        self.model = model
        self.historial = {}
    
    def ejecutar(self):
        self.model.fit(self.dataset_train.features(), self.dataset_train.targets())
        
        # Métricas en train
        y_true_train = self.dataset_train.targets()
        y_pred_train = self.model.predict(self.dataset_train.features())
        self.historial['train'] = {
            'MSE': MetricasRegresion.mse(y_true_train, y_pred_train),
            'MAE': MetricasRegresion.mae(y_true_train, y_pred_train),
            'R2': MetricasRegresion.r2_score(y_true_train, y_pred_train)
        }
        
        # Métricas en test
        y_true_test = self.dataset_test.targets()
        y_pred_test = self.model.predict(self.dataset_test.features())
        self.historial['test'] = {
            'MSE': MetricasRegresion.mse(y_true_test, y_pred_test),
            'MAE': MetricasRegresion.mae(y_true_test, y_pred_test),
            'R2': MetricasRegresion.r2_score(y_true_test, y_pred_test)
        }

    def reporte(self) -> str:
        reporte_str = f"Experimento: {self.nombre}\nModelo: {self.model.__class__.__name__}\n\nMétricas:\n"
        for dataset, metricas in self.historial.items():
            reporte_str += f"  {dataset.capitalize()}:\n"
            reporte_str += f"    MSE: {metricas['MSE']:.4f}\n"
            reporte_str += f"    MAE: {metricas['MAE']:.4f}\n"
            reporte_str += f"    R²: {metricas['R2']:.4f}\n"
        return reporte_str

# TODO: ejecutar experimento y mostrar reporte
experimento = ExperimentoRegresion("Regresión Lineal Simple", train_data, test_data, RegresionLinealSimple())
experimento.ejecutar()
print(experimento.reporte())
# TODO: inspeccionar historial
print("Historial:", experimento.historial)

---
---
## Parte 2 - Introduccion a NumPy y Matplotlib



In [ ]:
from __future__ import annotations

from random import Random
from time import perf_counter

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)


## Ejercicio 7 - Obtener arrays desde las clases de mediciones y dataset

1. Construir `temp_array` como array de `numpy` a partir de `serie_temp.valores()`.
2. Obtener `x_train`, `y_train`, `x_test`, `y_test` como arrays de numpy a partir de `train_ds` y `test_ds`. 
3. Construir `XY_train` como array de numpy de dimensión `(n_train, 2)` apilando `x_train` e `y_train` por columnas. Practicar `slicing`: para cada uno de los siguientes enunciados, crear una nueva variable a partir de `XY_train` (e imprimir su dimensión) que contenga:
   - los primeros 20 elementos de `x`.
   - los últimos 20 elementos de `y`.
   - la multiplicación de los dos arrays anteriores mediante `*`. Analizar qué significa esta operación. 
   - el logaritmo de los primeros 20 elementos de `x`. 
   - los pares `(x,y)` con índice par.
   - el vector `y`.
   - un subconjunto aleatorio de 25 elementos `(x,y)` de `xy_train`.
4. Imprimir para todos los arrays construídos: `shape`, `dtype`, `ndim`, `size`.
5. Verificar que:
   - la dimensión de `temp_array` es 1: `temp_array.ndim == 1`
   - los tamaños de `x_train` y `y_train` son iguales: `x_train.shape == y_train.shape`
   - La segunda dimensión de `XY_train` contiene dos entradas: `XY_train.shape[1] == 2`

In [ ]:


# Ejercicio 7

# 1. Construir temp_array como array de numpy a partir de serie_temp.valores()
temp_array = np.array(serie1.valores())

# 2. Obtener x_train, y_train, x_test, y_test como arrays de numpy a partir de train_ds y test_ds
x_train = np.array(train_data.features())
y_train = np.array(train_data.targets())
x_test = np.array(test_data.features())
y_test = np.array(test_data.targets())

# 3. Construir XY_train como array de numpy de dimensión (n_train, 2) apilando x_train e y_train por columnas
XY_train = np.column_stack((x_train, y_train))

# Practicar slicing
# - los primeros 20 elementos de x
primeros_20_x = XY_train[:20, 0]
print(f"Primeros 20 elementos de x: dimensión {primeros_20_x.shape}")

# - los últimos 20 elementos de y
ultimos_20_y = XY_train[-20:, 1]
print(f"Últimos 20 elementos de y: dimensión {ultimos_20_y.shape}")

# - la multiplicación de los dos arrays anteriores mediante *
multiplicacion = primeros_20_x * ultimos_20_y
print(f"Multiplicación: dimensión {multiplicacion.shape}")
print("Esta operación realiza una multiplicación elemento a elemento entre los primeros 20 x y los últimos 20 y.")

# - el logaritmo de los primeros 20 elementos de x
log_primeros_20_x = np.log(primeros_20_x)
print(f"Logaritmo de primeros 20 x: dimensión {log_primeros_20_x.shape}")

# - los pares (x,y) con índice par
pares_xy = XY_train[::2]
print(f"Pares (x,y) con índice par: dimensión {pares_xy.shape}")

# - el vector y
vector_y = XY_train[:, 1]
print(f"Vector y: dimensión {vector_y.shape}")

# - un subconjunto aleatorio de 25 elementos (x,y) de XY_train
subconjunto_aleatorio = XY_train[np.random.choice(XY_train.shape[0], 25, replace=False)]
print(f"Subconjunto aleatorio de 25 elementos: dimensión {subconjunto_aleatorio.shape}")

# 4. Imprimir para todos los arrays construídos: shape, dtype, ndim, size
arrays = {
    'temp_array': temp_array,
    'x_train': x_train,
    'y_train': y_train,
    'x_test': x_test,
    'y_test': y_test,
    'XY_train': XY_train,
    'primeros_20_x': primeros_20_x,
    'ultimos_20_y': ultimos_20_y,
    'multiplicacion': multiplicacion,
    'log_primeros_20_x': log_primeros_20_x,
    'pares_xy': pares_xy,
    'vector_y': vector_y,
    'subconjunto_aleatorio': subconjunto_aleatorio
}

for name, arr in arrays.items():
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}, ndim={arr.ndim}, size={arr.size}")

# 5. Verificar que:
# - la dimensión de temp_array es 1: temp_array.ndim == 1
assert temp_array.ndim == 1, f"temp_array.ndim es {temp_array.ndim}, debería ser 1"

# - los tamaños de x_train y y_train son iguales: x_train.shape == y_train.shape
assert x_train.shape == y_train.shape, f"x_train.shape {x_train.shape} != y_train.shape {y_train.shape}"

# - La segunda dimensión de XY_train contiene dos entradas: XY_train.shape[1] == 2
assert XY_train.shape[1] == 2, f"XY_train.shape[1] es {XY_train.shape[1]}, debería ser 2"

print("Todas las verificaciones pasaron correctamente.")

---
## Ejercicio 8 - Beneficio de usar NumPy
Vamos a comparar una operación equivalente hecha con listas de Python y con arrays de `numpy`, usando los datos anteriores.

1. Partir de `serie_temp.valores()` y construir una lista larga `temps_grandes_lista` repitiendo esas mediciones muchas veces (por ejemplo 200_000 repeticiones).
2. Construir `temps_grandes_array` como `np.array(temps_grandes_lista)`.
3. centrar los datos respecto al valor promedio (calcular y restar el valor promedio):
   - con listas Python
   - con NumPy
4. Medir tiempos con [`perf_counter()`](https://docs.python.org/3/library/time.html#module-time) para ambas variantes.
5. Verificar con [`np.allclose`](https://numpy.org/devdocs/reference/generated/numpy.allclose.html) que ambas producen el mismo resultado.
6. Reportar:
   - tiempo con listas
   - tiempo con NumPy
   - factor aproximado de mejora

In [ ]:
# TODO: crear temps_grandes_lista y temps_grandes_array
temps_grandes_lista = serie1.valores()[:5000]
temps_grandes_array = np.array(temps_grandes_lista)
# TODO: calcular la version centrada con listas
# temps_centrados_lista = [temp - np.mean(temps_grandes_lista) for temp in temps_grandes_lista]
# TODO: calcular la version centrada con NumPy
# temps_centrados_array = temps_grandes_array - np.mean(temps_grandes_array)
# TODO: medir tiempos y comparar resultados
start_time_lista = perf_counter()
temps_centrados_lista = [temp - np.mean(temps_grandes_lista) for temp in temps_grandes_lista]
end_time_lista = perf_counter()
tiempo_lista = end_time_lista - start_time_lista
start_time_array = perf_counter()
temps_centrados_array = temps_grandes_array - np.mean(temps_grandes_array)
end_time_array = perf_counter()
tiempo_array = end_time_array - start_time_array
print(f"Tiempo con listas: {tiempo_lista:.4f} segundos")
print(f"Tiempo con NumPy: {tiempo_array:.4f} segundos")

---

## Ejercicio 9 - Vectorización en NumPy

Ya tenemos `modelo` entrenado  en el Ejercicio 4. Ahora vamos a usar su pendiente e intercepto con numpy arrays.

1. Construir `y_hat_train_np` y `y_hat_test_np` usando la formula vectorizada:
$$
\hat{T} = m \cdot P + b
$$ 

donde `m` y `b` son la pendiente y el intercepto estimadas por el modelo y P el vector de entrenamiento en formato de array de numpy.

2. Obtener las predicciones del modelo para train y test.
3. Verificar que las predicciones vectorizadas y las del modelo coinciden.
4. Calcular el error cuadrático medio (MSE) para los conjuntos en entrenamiento y prueba en forma vectorizada.
5. Mostrar las primeras predicciones de ambos enfoques para train.

In [ ]:
# TODO: usar pendiente e intercepto de modelo sobre arrays
y_hat_train_np = model.coef_ * x_train + model.intercept_
y_hat_test_np = model.coef_ * x_test + model.intercept_
# TODO: comparar contra predict del modelo 
y_hat_train_predict = model.predict(x_train)
y_hat_test_predict = model.predict(x_test)
assert np.allclose(y_hat_train_np, y_hat_train_predict), "Las predicciones en train no coinciden"
assert np.allclose(y_hat_test_np, y_hat_test_predict), "Las predicciones en test no coinciden"
# TODO: calcular mse_train y mse_test vectorizados
mse_train = np.mean((y_train - y_hat_train_np) ** 2)
mse_test = np.mean((y_test - y_hat_test_np) ** 2)
# TODO: imprimir resultados
print(f"MSE Train: {mse_train:.4f}, MSE Test: {mse_test:.4f}")

---

## Ejercicio 10 - Reobtener el mismo ajuste con álgebra lineal de NumPy

Veamos la facilidades provistas por numpy. Queremos reproducir el ajuste lineal usando `numpy.linalg.lstsq`.


1. Construir la matriz de diseño `A_train = [x_train, 1]`.
2. Resolver el problema de mínimos cuadrados con [`np.linalg.lstsq`](https://numpy.org/devdocs/reference/generated/numpy.linalg.lstsq.html).
3. Obtener `m_hat_np` y `b_hat_np`.
4. Comparar esos parámetros con `modelo.pendiente` y `modelo.intercepto` de los ejercicios anteriores.
5. Verificar con `np.allclose` que ambos ajustes son equivalentes dentro de una tolerancia razonable.
6. Calcular `y_hat_test_lstsq` y el `mae_test_lstsq`.


In [ ]:
# TODO: armar A_train, útil: np.ones_like 
A_train = np.column_stack((x_train, np.ones_like(x_train)))
# TODO: resolver lstsq
params = np.linalg.lstsq(A_train, y_train, rcond=None)[0]
# TODO: comparar parámetros con el ajuste de RegresionLinealSimple
print(f"Parámetros obtenidos con lstsq: {params}")
print(f"Parámetros del modelo RegresionLinealSimple: coeficiente={model.coef_}, intercepto={model.intercept_}")
# TODO: calcular predicciones y MAE en test
A_test = np.column_stack((x_test, np.ones_like(x_test)))
y_hat_test_lstsq = A_test @ params
mae_test_lstsq = np.mean(np.abs(y_test - y_hat_test_lstsq))
print(f"MAE en test con lstsq: {mae_test_lstsq:.4f}")


---

## Ejercicio 11 - Visualizar datos y ajuste con Matplotlib

Explorar la capacidad de la librería [`matplotlib`](https://matplotlib.org/) para graficar.

1. Graficar los puntos de train y test en un mismo gráfico con colores distintos.
2. Superponer la recta del ajuste obtenido por el modelo de regresión inicial.
3. Agregar título, ejes y leyenda.
4. En una segunda figura, graficar los residuos en el conjunto de prueba:
$$
r_i = y_i - \hat{y}_i
$$
5. Verificar visualmente si el ajuste parece razonable.


In [ ]:
# TODO: scatter de train y test
plt.figure(figsize=(7, 4))
plt.scatter(x_train, y_train, alpha=0.7, label='Train')
plt.scatter(x_test, y_test, alpha=0.7, label='Test')

# TODO: recta del ajuste de POO
x_line = np.linspace(min(dataset.features()), max(dataset.features()), 100)
y_line = model.predict(x_line)
plt.plot(x_line, y_line, color='red', label='Regresión Lineal')
plt.xlabel('Presión (hPa)')
plt.ylabel('Temperatura (°C)')
plt.title('Datos de Entrenamiento y Prueba')
plt.legend()
plt.grid(True)
plt.show()
# TODO: grafico de residuos en test
residuos_test = y_test - y_hat_test_np
plt.figure(figsize=(7, 4))
plt.scatter(x_test, residuos_test, alpha=0.7)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Presión (hPa)')
plt.ylabel('Residuo (°C)')
plt.title('Gráfico de Residuos en Test')
plt.grid(True)
plt.show()

---
---

## Parte 3 - Introduccion a PyTorch

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

---

## Ejercicio 12 — Tensores: creación, dtypes, shapes y dispositivos


1. Crear los siguientes tensores:
   - `t0`: tensor 1D con valores [100, 0, 25.6] como (`float32`) usando `torch.tensor`
   - `a`: tensor 1D con valores `[0,1,2,...,9]` (`int64`) usando `torch.arange`
   - `b`: tensor 2D de shape `(3,4)` con valores aleatorios uniformes en `[0,1)` (`float32`) utilizando `torch.rand`
   - `c`: tensor 3D de shape `(2,3,4)` lleno de unos utilizando `torch.ones`
2. Crear una función llamada `mostrar_info` que reciba un tensor e imprima su
   - `dtype`, `shape`, `device`, `requires_grad`

   Utilizar la función para cada tensor creado anteriormente.

3. Convertir `a` a `float32` utilizando el método de los tensores `.to()` y normalizarlo a media 0 y desvío 1 utilizando los métodos `.mean()` y `.std()`.
4. Mover `b` a GPU **si tiene GPU disponible** utilizando el método `.to()`; si no, dejarlo en CPU.

In [ ]:
# TODO: crear t0, a, b, c
# TODO: imprimir propiedades
# TODO: convertir/normalizar a
# TODO: mover b a CUDA si existe

---

## Ejercicio 13 — Indexing, slicing, broadcasting y operaciones básicas

1. Crear un tensor `X` con valores `0,1,...,29` y reacomodarlo a dimensión `(5, 6)` utilizando el método `.view()` de los tensores de pytorch.
2. Extraer:
   - la tercera fila (índice 2) completa
   - la quinta columna (índice 4) completa
   - un sub-bloque de 3x2 (filas 1..3, columnas 2..3)
3. Crear un vector `v` de dimensión `(6,)` y sumar `X + v` usando broadcasting.
4. Verificar con asserts:
   - que `X + v` tiene shape `(5,6)`
   - que la fila 0 de `X+v` coincide con `X[0] + v`
5. crear un tensor con elementos enteros aleatorios entre -10 y 10, de dimensión (10,8,2)

In [ ]:
# TODO: crear X con shape (5,6)
# TODO: slicing solicitado
# TODO: broadcasting con v
# TODO: asserts
# TODO(opcional): view vs reshape vs transpose

---

## Ejercicio 14 — Autograd: gradientes y verificación numérica simple

1. Definir un tensor `x` escalar con `requires_grad=True` e inicializarlo con algún valor (ej. 1.5).
2. Definir la función:
$$
f(x) = x^3 + 2x^2 - 5x + 1
$$
3. Calcular `f(x)` y obtener `df/dx` usando `backward()`.
4. Verificar aproximadamente el gradiente usando diferencias finitas:

$$
f'(x) \approx \frac{f(x+\epsilon) - f(x-\epsilon)}{2\epsilon}
$$

con `epsilon = 1e-4`.

5. Reportar el error absoluto entre autograd y diferencias finitas.

In [ ]:
# TODO: definir x con requires_grad=True
# TODO: definir f(x) y backward
# TODO: gradiente por diferencias finitas
# TODO: comparar y reportar error absoluto

---

## Ejercicio 15 — `nn.Module`: construir un modelo y contar parámetros

1. Implementar una clase `MLP` (heredando de `nn.Module`) con:
   - `Linear(in_features=10, out_features=32)`
   - `ReLU`
   - `Linear(32, 1)`
2. Crear una instancia del modelo y:
   - imprimir el modelo
   - contar parámetros entrenables (total) y mostrar el número
3. Hacer un paso hacia adelante de la red (forward pass) con un batch sintético `X` de shape `(16, 10)` y chequear que la salida tiene shape `(16, 1)`.
4. Explicar brevemente (Markdown): diferencia entre `model.parameters()` y `state_dict()`.

In [ ]:

# TODO: definir MLP(nn.Module)
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        
        layers = []
        layers.append(nn.Linear(10, 32))
        layers.append(nn.ReLU())
        layers.append(nn.Linear(32, 1))
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)

# TODO: instanciar, imprimir, contar parámetros
model = MLP()
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")
# TODO: forward con batch sintético y assert de shape
batch_size = 16
input_tensor = torch.randn(batch_size, 10)
output_tensor = model(input_tensor)
assert output_tensor.shape == (batch_size, 1), f"Output shape es {output_tensor.shape}, debería ser {(batch_size, 1)}"
# TODO: breve explicación en Markdown (en celda aparte)
# TODO: forward con batch sintético y assert de shape
# TODO: breve explicación en Markdown (en celda aparte)

model.parameters() returns an iterator over the model's learnable parameters (tensors that require gradients, such as weights and biases in layers). It's used for operations like counting parameters or passing to optimizers.

state_dict() returns a dictionary containing the entire state of the model, including learnable parameters, buffers (e.g., running means in BatchNorm), and other non-parameter tensors. It's designed for saving and loading model checkpoints, preserving the model's structure and values.

In summary, parameters() focuses on trainable tensors for computation, while state_dict() captures the full serializable state for persistence. For example, state_dict() includes buffers that parameters() excludes.

---

## Ejercicio 16 — Entrenamiento básico: loop, `optim`, `train()`/`eval()`


Vamos a entrenar un modelo simple en un dataset sintético de regresión:
$$
y = 3x + 0.5 + \text{ruido}
$$
donde \(x\) es 1D.


1. Generar datos sintéticos:
   - `N=512` puntos
   - `x` uniforme en `[-2, 2]`
   - ruido Gaussiano con `std=0.2`
2. Definir un modelo `nn.Linear(1,1)`.
3. Elegir:
   - pérdida MSE (`nn.MSELoss`)
   - optimizador SGD (`torch.optim.SGD`) con `lr` razonable
4. Implementar loop de entrenamiento por `epochs=300`:
   - usar `model.train()`
   - `zero_grad()`, forward, loss, `backward()`, `step()`
   - loggear la loss cada 50 épocas
5. Al final, imprimir los parámetros aprendidos (`weight`, `bias`) y compararlos con (3, 0.5).

4. (Opcional) correr un paso con `model.eval()` y `torch.no_grad()` para predecir 5 valores.

In [ ]:
import torch 
from torch import nn
# TODO: generar dataset sintético (x, y) como tensores float32 con shape (N,1)
N = 512
#x uniforme entre -2 y 2, y = 3x + 1 + ruido gaussiano
x = torch.rand((N, 1), dtype=torch.float32) * 4 - 2  # Uniforme entre -2 y 2
ruido = torch.randn((N, 1), dtype=torch.float32) * 0.5  # Ruido gaussiano con desviación estándar de 0.5
y = 3 * x + 0.5 + ruido
# TODO: definir modelo nn.Linear(1,1), loss y optimizer
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
# TODO: loop de entrenamiento con logs
num_epochs = 500
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    y_pred = model(x)
    loss = loss_fn(y_pred, y)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")
# TODO: imprimir weight/bias aprendidos y comparar
print(f"Weight: {model.weight.item():.4f}")
print(f"Bias: {model.bias.item():.4f}")

import matplotlib.pyplot as plt

plt.scatter(x.detach().numpy(), y.detach().numpy(), alpha=0.5)
plt.plot(x.detach().numpy(), model(x).detach().numpy(), color='red')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Aprendizaje de una regresión lineal')
plt.show()

---

## Ejercicio 17 — `Dataset` y `DataLoader`: batching, shuffle y collation

1. Crear una clase `ToyDataset(Dataset)` que:
   - reciba tensores `X` e `y`
   - implemente `__len__` y `__getitem__`
2. Reutilizar los datos sintéticos del Ejercicio 16 y crear:
   - `dataset = ToyDataset(X, y)`
   - `loader = DataLoader(dataset, batch_size=64, shuffle=True)`
3. Iterar sobre `loader` y:
   - imprimir la dimensión del primer batch
   - verificar que `X_batch` es `(64,1)` y `y_batch` es `(64,1)`
4. Modificar el loop de entrenamiento del Ejercicio 16 para usar el `DataLoader` (mini-batches).


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# TODO: definir ToyDataset
class ToyDataset(Dataset):
    def __init__(self, x: torch.Tensor, y: torch.Tensor):
        self.x = x
        self.y = y
    
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]
# TODO: crear dataset y dataloader
dataset = ToyDataset(x, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)
# TODO: iterar un batch y verificar shapes
for batch_x, batch_y in loader:
    print(f"Batch x shape: {batch_x.shape}, Batch y shape: {batch_y.shape}")
    break
# TODO: re-entrenar el modelo usando mini-batches desde el loader
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
num_epochs = 500
for epoch in range(num_epochs):
    model.train()
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        y_pred = model(batch_x)
        loss = loss_fn(y_pred, batch_y)
        loss.backward()
        optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")

print(f"Weight: {model.weight.item():.4f}")
print(f"Bias: {model.bias.item():.4f}")

---

## Ejercicio 18 — Guardar y cargar: `state_dict`, checkpoints y reproducibilidad

### Consignas
1. Después de entrenar el modelo del ejercicio 17, guardar en un diccionario un checkpoint que incluya:
   - `model_state_dict`
   - `optimizer_state_dict`
   - `epoch`
   - `train_loss` final (un float)

2. Guardar en un archivo: `checkpoint.pt`.
3. Crear un nuevo modelo (misma arquitectura) y un nuevo optimizador.
4. Cargar el checkpoint y restaurar estados.
5. Verificar que, con el modelo restaurado en `eval()` y `no_grad()`, la predicción para un mismo `x_test` coincide (o es extremadamente cercana) con la del modelo original.

**Nota:** usar `torch.save` / `torch.load` y `load_state_dict`.

In [ ]:
from pathlib import Path
import tempfile

# TODO: armar diccionario checkpoint y guardar a un archivo escribible
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': num_epochs,
    'loss': loss.item(),
}

checkpoint_path = Path(tempfile.gettempdir()) / 'checkpoint.pt'
torch.save(checkpoint, checkpoint_path)
print(f"Checkpoint guardado en: {checkpoint_path}")

# TODO: reinstanciar modelo/optimizer y cargar checkpoint
restored_model = nn.Linear(1, 1)
restored_optimizer = torch.optim.SGD(restored_model.parameters(), lr=0.01)
loaded_checkpoint = torch.load(checkpoint_path, map_location='cpu')
restored_model.load_state_dict(loaded_checkpoint['model_state_dict'])
restored_optimizer.load_state_dict(loaded_checkpoint['optimizer_state_dict'])

# TODO: verificar predicciones equivalentes en un x_test fijo
x_test = torch.tensor([[0.0]], dtype=torch.float32)
model.eval()
restored_model.eval()
with torch.no_grad():
    original_pred = model(x_test)
    restored_pred = restored_model(x_test)

print(f"Predicción original: {original_pred.item():.6f}")
print(f"Predicción restaurada: {restored_pred.item():.6f}")
assert torch.allclose(original_pred, restored_pred), 'Las predicciones no coinciden tras restaurar el checkpoint'

---

## Ejercicio 19 — Modelado de la ley de enfriamiento mediante una red neuronal

En este ejercicio aproximaremos la ley de enfriamiento de Netwon con una red neuronal. Ésta describe cómo varía la temperatura de un objeto en función del tiempo cuando se encuentra en un ambiente con temperatura constante. La ecuación del proceso es 

$$
T(t) = T_{amb} + (T_0 - T_{amb}) \exp(-kt),
$$

en donde $T(t)$ es la temperatura al tiempo $t$, $T_0$ la temperatura inicial, $T_{amb}$ la ambiente y $k$ la constante de enfriamiento.

Se busca entrenar un modelo de red neuronal (tipo MLP) que aprenda a aproximar esta función a partir de datos simulados con ruido. 

1. Implementar una función que genere datos sintéticos a partir de la ley de enfriamiento de Newton. Agregar ruido gaussiano a las observaciones para simular mediciones reales y generar una base de datos de 1000 puntos. Visualizar los datos en un gráfico de dispersión.

2. Dividir los datos en entrenamiento y prueba y generar un dataloader con tamaño de batch de 32.

3. Implementar el modelo MLP en pytorch utilizando `torch.nn.Module`. El modelo debe recibir como entrada el tiempo $t$ y devolver la temperatura estimada. Utilizar al menos una capa oculta y una función de activación no lineal. 

5. Definir una función de pérdida adecuada para regresión y un optimizador utilizar una tasa de aprendizaje de $0.002$. Implementar un loop de entrenamiento que haga un forward pass, calcule la pérdida, haga un backward pass, y actualice los parámetros. Entrenar el modelo con los datos de entrenamiento durante 4000 épocas

4. Mostrar el valor de la pérdida cada cierta cantidad de épocas y analizar si el modelo está aprendiendo correctamente. 

5. Comparar visualmente las predicciones del modelo con los datos de prueba. Graficar la curva de pérdida.

In [ ]:
import copy
import itertools
import torch

# Reproducibilidad
torch.manual_seed(42)

# 1) Generación de datos sintéticos (Ley de enfriamiento de Newton + ruido)
T0 = 100.0
Tamb = 25.0
k = 0.05
t_max = 100.0
n_puntos = 1000

t = torch.linspace(0, t_max, n_puntos, dtype=torch.float32)
temperatura = Tamb + (T0 - Tamb) * torch.exp(-k * t)
ruido = torch.randn_like(temperatura) * 2.0

y_obs = (temperatura + ruido).unsqueeze(1)
X_obs = t.unsqueeze(1)

# 2) Split train/val/test y dataloaders
dataset = ToyDataset(X_obs, y_obs)
n_total = len(dataset)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# 3) Modelo configurable
class MLPEnfriamiento(nn.Module):
    def __init__(self, hidden_dim: int = 64, activation: str = "relu"):
        super().__init__()
        act = nn.ReLU() if activation == "relu" else nn.Tanh()
        self.layers = nn.Sequential(
            nn.Linear(1, hidden_dim),
            act,
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.layers(x)


def build_loss(loss_name: str):
    if loss_name == "mse":
        return nn.MSELoss()
    if loss_name == "huber":
        return nn.HuberLoss()
    if loss_name == "mae":
        return nn.L1Loss()
    raise ValueError(f"Loss desconocida: {loss_name}")


def build_optimizer(opt_name: str, params, lr: float):
    if opt_name == "adam":
        return torch.optim.Adam(params, lr=lr)
    if opt_name == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr)
    if opt_name == "sgd":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    raise ValueError(f"Optimizador desconocido: {opt_name}")


def run_epoch(model, loader, loss_fn, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    total_n = 0
    for batch_x, batch_y in loader:
        if training:
            optimizer.zero_grad()

        pred = model(batch_x)
        loss = loss_fn(pred, batch_y)

        if training:
            loss.backward()
            optimizer.step()

        bs = batch_x.size(0)
        total_loss += loss.item() * bs
        total_n += bs

    return total_loss / total_n


def train_config(config, max_epochs=1200, patience=120, min_delta=1e-5):
    model = MLPEnfriamiento(hidden_dim=config["hidden_dim"], activation=config["activation"])
    loss_fn = build_loss(config["loss"])
    optimizer = build_optimizer(config["optimizer"], model.parameters(), config["lr"])

    history_train = []
    history_val = []

    best_val = float("inf")
    best_epoch = -1
    best_state = None
    wait = 0

    for epoch in range(max_epochs):
        train_loss = run_epoch(model, train_loader, loss_fn, optimizer)
        val_loss = run_epoch(model, val_loader, loss_fn, optimizer=None)

        history_train.append(train_loss)
        history_val.append(val_loss)

        if val_loss < (best_val - min_delta):
            best_val = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            break

    # Restaurar mejor checkpoint de esta configuración
    model.load_state_dict(best_state)

    # Métricas en test
    test_mse = run_epoch(model, test_loader, nn.MSELoss(), optimizer=None)
    test_mae = run_epoch(model, test_loader, nn.L1Loss(), optimizer=None)

    return {
        "config": config,
        "model": model,
        "best_val_loss": best_val,
        "best_epoch": best_epoch,
        "test_mse": test_mse,
        "test_mae": test_mae,
        "history": {
            "train": history_train,
            "val": history_val,
        },
    }


# 4) Búsqueda automática de mejor combinación (modelo + loss + optimizador)
search_space = {
    "hidden_dim": [32, 64, 128],
    "activation": ["relu", "tanh"],
    "loss": ["mse", "huber", "mae"],
    "optimizer": ["adam", "rmsprop", "sgd"],
    "lr": [0.002, 0.001],
}

keys = list(search_space.keys())
configs = [dict(zip(keys, values)) for values in itertools.product(*(search_space[k] for k in keys))]

results = []
for i, cfg_run in enumerate(configs, start=1):
    out = train_config(cfg_run)
    results.append(out)

    if i % 10 == 0 or i == len(configs):
        print(f"Completadas {i}/{len(configs)} configuraciones")

# Ordenar por validación y elegir mejor
results_sorted = sorted(results, key=lambda r: r["best_val_loss"])
best_result = results_sorted[0]
best_model = best_result["model"]

print("\nMejor configuración encontrada:")
print(best_result["config"])
print(f"Best val loss: {best_result['best_val_loss']:.6f} (epoch {best_result['best_epoch']})")
print(f"Test MSE: {best_result['test_mse']:.6f}")
print(f"Test MAE: {best_result['test_mae']:.6f}")

# 5) Predicción con el mejor modelo
best_model.eval()
with torch.no_grad():
    t_test = torch.linspace(0, t_max, n_puntos).unsqueeze(1)
    pred_best = best_model(t_test)

plt.figure(figsize=(10, 6))
plt.scatter(t.numpy(), (temperatura + ruido).numpy(), alpha=0.35, label="Datos con ruido")
plt.plot(t_test.numpy(), temperatura.numpy(), color="black", linewidth=2, linestyle="--", label="Ley verdadera")
plt.plot(t_test.numpy(), pred_best.numpy(), color="red", linewidth=2, label="Mejor MLP")
plt.xlabel("Tiempo")
plt.ylabel("Temperatura")
plt.title("Mejor modelo tras búsqueda automática")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import copy
import itertools
import torch

# Reproducibilidad
torch.manual_seed(42)

# 1) Generación de datos sintéticos (Ley de enfriamiento de Newton + ruido)
T0 = 100.0
Tamb = 25.0
k = 0.05
t_max = 100.0
n_puntos = 1000

t = torch.linspace(0, t_max, n_puntos, dtype=torch.float32)
temperatura = Tamb + (T0 - Tamb) * torch.exp(-k * t)
ruido = torch.randn_like(temperatura) * 2.0

y_obs = (temperatura + ruido).unsqueeze(1)
X_obs = t.unsqueeze(1)

# 2) Split train/val/test y dataloaders
dataset = ToyDataset(X_obs, y_obs)
n_total = len(dataset)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# 3) Modelo configurable
class MLPEnfriamiento(nn.Module):
    def __init__(self, hidden_dim: int = 64, activation: str = "relu"):
        super().__init__()
        act = nn.ReLU() if activation == "relu" else nn.Tanh()
        self.layers = nn.Sequential(
            nn.Linear(1, hidden_dim),
            act,
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.layers(x)


def build_loss(loss_name: str):
    if loss_name == "mse":
        return nn.MSELoss()
    if loss_name == "huber":
        return nn.HuberLoss()
    if loss_name == "mae":
        return nn.L1Loss()
    raise ValueError(f"Loss desconocida: {loss_name}")


def build_optimizer(opt_name: str, params, lr: float):
    if opt_name == "adam":
        return torch.optim.Adam(params, lr=lr)
    if opt_name == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr)
    if opt_name == "sgd":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    raise ValueError(f"Optimizador desconocido: {opt_name}")


def run_epoch(model, loader, loss_fn, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    total_n = 0
    for batch_x, batch_y in loader:
        if training:
            optimizer.zero_grad()

        pred = model(batch_x)
        loss = loss_fn(pred, batch_y)

        if training:
            loss.backward()
            optimizer.step()

        bs = batch_x.size(0)
        total_loss += loss.item() * bs
        total_n += bs

    return total_loss / total_n


def train_config(config, max_epochs=1200, patience=120, min_delta=1e-5):
    model = MLPEnfriamiento(hidden_dim=config["hidden_dim"], activation=config["activation"])
    loss_fn = build_loss(config["loss"])
    optimizer = build_optimizer(config["optimizer"], model.parameters(), config["lr"])

    history_train = []
    history_val = []

    best_val = float("inf")
    best_epoch = -1
    best_state = None
    wait = 0

    for epoch in range(max_epochs):
        train_loss = run_epoch(model, train_loader, loss_fn, optimizer)
        val_loss = run_epoch(model, val_loader, loss_fn, optimizer=None)

        history_train.append(train_loss)
        history_val.append(val_loss)

        if val_loss < (best_val - min_delta):
            best_val = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            break

    # Restaurar mejor checkpoint de esta configuración
    model.load_state_dict(best_state)

    # Métricas en test
    test_mse = run_epoch(model, test_loader, nn.MSELoss(), optimizer=None)
    test_mae = run_epoch(model, test_loader, nn.L1Loss(), optimizer=None)

    return {
        "config": config,
        "model": model,
        "best_val_loss": best_val,
        "best_epoch": best_epoch,
        "test_mse": test_mse,
        "test_mae": test_mae,
        "history": {
            "train": history_train,
            "val": history_val,
        },
    }


# 4) Búsqueda automática de mejor combinación (modelo + loss + optimizador)
search_space = {
    "hidden_dim": [128, 200],
    "activation": ["relu"],
    "loss": ["huber"],
    "optimizer": ["adam", "rmsprop", "sgd"],
    "lr": [0.002, 0.001],
}

keys = list(search_space.keys())
configs = [dict(zip(keys, values)) for values in itertools.product(*(search_space[k] for k in keys))]

results = []
for i, cfg_run in enumerate(configs, start=1):
    out = train_config(cfg_run)
    results.append(out)

    if i % 10 == 0 or i == len(configs):
        print(f"Completadas {i}/{len(configs)} configuraciones")

# Ordenar por validación y elegir mejor
results_sorted = sorted(results, key=lambda r: r["best_val_loss"])
best_result = results_sorted[0]
best_model = best_result["model"]

print("\nMejor configuración encontrada:")
print(best_result["config"])
print(f"Best val loss: {best_result['best_val_loss']:.6f} (epoch {best_result['best_epoch']})")
print(f"Test MSE: {best_result['test_mse']:.6f}")
print(f"Test MAE: {best_result['test_mae']:.6f}")

# 5) Predicción con el mejor modelo
best_model.eval()
with torch.no_grad():
    t_test = torch.linspace(0, t_max, n_puntos).unsqueeze(1)
    pred_best = best_model(t_test)

plt.figure(figsize=(10, 6))
plt.scatter(t.numpy(), (temperatura + ruido).numpy(), alpha=0.35, label="Datos con ruido")
plt.plot(t_test.numpy(), temperatura.numpy(), color="black", linewidth=2, linestyle="--", label="Ley verdadera")
plt.plot(t_test.numpy(), pred_best.numpy(), color="red", linewidth=2, label="Mejor MLP")
plt.xlabel("Tiempo")
plt.ylabel("Temperatura")
plt.title("Mejor modelo tras búsqueda automática")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Curvas de pérdida de las mejores configuraciones (top-3 por val loss)
plt.figure(figsize=(11, 6))

top_k = 3
for rank, r in enumerate(results_sorted[:top_k], start=1):
    cfg = r["config"]
    name = (
        f"#{rank} h={cfg['hidden_dim']} act={cfg['activation']} "
        f"loss={cfg['loss']} opt={cfg['optimizer']} lr={cfg['lr']}"
    )
    plt.semilogy(r["history"]["train"], alpha=0.45, label=f"Train {name}")
    plt.semilogy(r["history"]["val"], linestyle="--", linewidth=2, label=f"Val {name}")

plt.xlabel("Época")
plt.ylabel("Pérdida")
plt.title("Comparación de entrenamiento/validación (top-3 configuraciones)")
plt.legend(fontsize=8)
plt.grid(True)
plt.show()